In [1]:
import pandas as pd
import os
import logging
from sqlalchemy import create_engine
import time
import urllib

In [7]:
# Create logs folder if it doesn't exist
os.makedirs('../logs', exist_ok=True)

# Logging Setup
logging.basicConfig(
    filename='../logs/ingest_db.log',
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filemode='a'
)

print("✅ Logging setup done!")

✅ Logging setup done!


In [8]:
import pyodbc

conn_str = (
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=Wasif\\SQLEXPRESS;'
    'DATABASE=InventoryDB;'
    'Trusted_Connection=yes;'
    'TrustServerCertificate=yes;'
)

conn = pyodbc.connect(conn_str)
print("✅ Connection Successful!")
conn.close()

✅ Connection Successful!


In [9]:
import urllib
from sqlalchemy import create_engine

params = urllib.parse.quote_plus(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=Wasif\\SQLEXPRESS;'
    'DATABASE=InventoryDB;'
    'Trusted_Connection=yes;'
    'TrustServerCertificate=yes;'
)

engine = create_engine(
    f'mssql+pyodbc:///?odbc_connect={params}',
    fast_executemany=True
)

print("✅ Engine Created Successfully!")

✅ Engine Created Successfully!


In [10]:
import os

# Point to the data folder
DATA_FOLDER = '../data'

# Check all CSV files are visible
for file in os.listdir(DATA_FOLDER):
    if '.csv' in file:
        print(f"Found: {file}")

Found: begin_inventory.csv
Found: end_inventory.csv
Found: purchases.csv
Found: purchase_prices.csv
Found: sales.csv
Found: vendor_invoice.csv


In [11]:
import time

def ingest_db(df, table_name, engine):
    """Inserts a dataframe into SQL Server table"""
    df.to_sql(
        name=table_name,
        con=engine,
        if_exists='replace',
        index=False,
        chunksize=1000
    )

print("✅ Function ready!")

✅ Function ready!


In [12]:
# Start with smallest file first - vendor_invoice
start = time.time()

file = 'vendor_invoice.csv'
table_name = 'vendor_invoice'

print(f"Loading {file}...")
df = pd.read_csv(os.path.join(DATA_FOLDER, file), low_memory=False)
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")

ingest_db(df, table_name, engine)

end = time.time()
print(f"✅ Done! Time taken: {(end-start):.2f} seconds")

Loading vendor_invoice.csv...
Rows: 5,543  |  Columns: 10
✅ Done! Time taken: 0.72 seconds


In [13]:
# Load purchase_prices
start = time.time()
file = 'purchase_prices.csv'
df = pd.read_csv(os.path.join(DATA_FOLDER, file), low_memory=False)
print(f"Loading {file}... Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
ingest_db(df, 'purchase_prices', engine)
print(f"✅ Done! Time: {(time.time()-start):.2f} seconds")

# Load begin_inventory
start = time.time()
file = 'begin_inventory.csv'
df = pd.read_csv(os.path.join(DATA_FOLDER, file), low_memory=False)
print(f"Loading {file}... Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
ingest_db(df, 'begin_inventory', engine)
print(f"✅ Done! Time: {(time.time()-start):.2f} seconds")

# Load end_inventory
start = time.time()
file = 'end_inventory.csv'
df = pd.read_csv(os.path.join(DATA_FOLDER, file), low_memory=False)
print(f"Loading {file}... Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
ingest_db(df, 'end_inventory', engine)
print(f"✅ Done! Time: {(time.time()-start):.2f} seconds")

Loading purchase_prices.csv... Rows: 12,261 | Columns: 9
✅ Done! Time: 0.54 seconds
Loading begin_inventory.csv... Rows: 206,529 | Columns: 9
✅ Done! Time: 9.37 seconds
Loading end_inventory.csv... Rows: 224,489 | Columns: 9
✅ Done! Time: 10.76 seconds


In [14]:
# Load purchases
start = time.time()
file = 'purchases.csv'
df = pd.read_csv(os.path.join(DATA_FOLDER, file), low_memory=False)
print(f"Loading {file}... Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
ingest_db(df, 'purchases', engine)
print(f"✅ Done! Time: {(time.time()-start):.2f} seconds")

Loading purchases.csv... Rows: 2,372,474 | Columns: 16
✅ Done! Time: 142.61 seconds


In [15]:
# Load sales
start = time.time()
file = 'sales.csv'
df = pd.read_csv(os.path.join(DATA_FOLDER, file), low_memory=False)
print(f"Loading {file}... Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
ingest_db(df, 'sales', engine)
print(f"✅ Done! Time: {(time.time()-start):.2f} seconds")

Loading sales.csv... Rows: 12,825,363 | Columns: 14
✅ Done! Time: 700.10 seconds


In [1]:
import logging
import os

os.makedirs('../logs', exist_ok=True)
logging.basicConfig(
    filename='../logs/ingest_db.log',
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filemode='a'
)

logging.info('=== Ingestion Started ===')
logging.info('Inserted: vendor_invoice - Rows: 5543')
logging.info('Inserted: purchase_prices - Rows: 12261')
logging.info('Inserted: begin_inventory - Rows: 206529')
logging.info('Inserted: end_inventory - Rows: 224489')
logging.info('Inserted: purchases - Rows: 2372474')
logging.info('Inserted: sales - Rows: 12825363')
logging.info('=== Ingestion Complete ===')

print("✅ Log updated!")

✅ Log updated!
